In [1]:
import numpy as np
import random
import itertools
from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import KFold

# ----------------------------------------------------------------------
# obtain center of grid
# ----------------------------------------------------------------------
def generate_grid_center(a, b, cell_size, N):
    point = np.arange(a, b, cell_size)
    centers = []
    center = [x + cell_size / 2 for x in point]
    for c in center:
        if (a < c < b):
            centers.append(c)
    point_centers = list(itertools.product(centers, repeat=N))
    return point_centers


Nz, Ny = 2, 4
a, b = 0, 32
rho_y, rho_z = 1, 1

# Generate grid and collect centers
centersy = generate_grid_center(a, b, rho_y, Ny)
centersz = generate_grid_center(a, b, rho_z, Nz)

# interconnection map (ground-truth M, here max(.), used only to label
# training data and to numerically validate eq. (18) below)
true_map = max

Data_sety = [(c, true_map(c)) for c in centersy]
Data_setz = [(c, true_map(c)) for c in centersz]

Xy, z = zip(*Data_sety)
Xz, w = zip(*Data_setz)

lambdas = np.logspace(-3, 3, 9)
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

lasso_cvy = LassoCV(alphas=lambdas, cv=kf)
lasso_cvy.fit(Xy, z)
lasso_cvz = LassoCV(alphas=lambdas, cv=kf)
lasso_cvz.fit(Xz, w)

best_lambday = lasso_cvy.alpha_
best_lambdaz = lasso_cvz.alpha_

lassoy = Lasso(alpha=best_lambday)
lassoy.fit(Xy, z)
lassoz = Lasso(alpha=best_lambdaz)
lassoz.fit(Xz, w)

My = lassoy.coef_   # \bar{M} for the y-subsystem (row of the sparse fit)
Mz = lassoz.coef_   # \bar{M} for the z-subsystem

print("My:", My)
print("Mz:", Mz)


# ----------------------------------------------------------------------
# Eq. (18):  \hat{\varepsilon}(\rho_x, w_l, x') := L_M ||\rho_x|| + ||w_l - \bar{M} x'||
#
# Convention: ||.|| is the sup-norm (consistent with rho_x being applied
# identically to every coordinate of the cell, and with L_M=1 being a
# valid Lipschitz constant for max(.) under the sup-norm). Pass
# ord=2 / ord=1 instead if your paper's norm differs.
# ----------------------------------------------------------------------
def eps_hat(rho_x, w_l, x_prime, M_bar, L_M, ord=np.inf):
    x_prime = np.atleast_1d(x_prime)
    rho_x = np.atleast_1d(rho_x)
    fit_residual = np.atleast_1d(w_l - np.dot(M_bar, x_prime))
    return L_M * np.linalg.norm(rho_x, ord=ord) + np.linalg.norm(fit_residual, ord=ord)


def verify_eps_bound(data_set, M_bar, rho_x, L_M, true_map, N,
                      n_random=200, ord=np.inf, seed=0, label="",
                      max_cells=500):
    r"""
    For (up to `max_cells` randomly-subsampled) grid cells (x_l, w_l) in
    `data_set`, checks that
        ||M(x') - \bar{M} x'|| <= \hat{\varepsilon}(\rho_x, w_l, x')
    holds for:
      (a) the 2^N worst-case box corners x_l +/- rho_x * e_i, and
      (b) n_random points drawn uniformly at random inside the box
          Phi_{rho_x}(x_l) = {x' : ||x' - x_l||_inf <= rho_x}.
    Returns the worst-case slack (rhs - lhs); a NEGATIVE value means
    the bound was violated for that sample. `max_cells` keeps this
    tractable for high-dimensional grids (e.g. Ny=4 -> 32^4 cells)
    without needing to sweep every single one to validate the bound.
    r"""
    rng = np.random.default_rng(seed)
    rho_vec = np.full(N, rho_x, dtype=float)
    worst_slack = np.inf
    n_checked = 0
    n_violations = 0
    eps_min = np.inf
    eps_max = -np.inf

    corners = list(itertools.product([-1.0, 1.0], repeat=N))

    cells = data_set
    if len(data_set) > max_cells:
        idx = rng.choice(len(data_set), size=max_cells, replace=False)
        cells = [data_set[i] for i in idx]

    for x_l, w_l in cells:
        x_l = np.asarray(x_l, dtype=float)

        # (a) worst-case corners of the box
        test_points = [x_l + np.array(c) * rho_vec for c in corners]
        # (b) random interior points
        test_points += [
            x_l + rng.uniform(-1.0, 1.0, size=N) * rho_vec
            for _ in range(n_random)
        ]

        for x_prime in test_points:
            lhs = np.abs(true_map(x_prime) - np.dot(M_bar, x_prime))
            rhs = eps_hat(rho_vec, w_l, x_prime, M_bar, L_M, ord=ord)
            slack = rhs - lhs
            n_checked += 1
            if slack < 0:
                n_violations += 1
            worst_slack = min(worst_slack, slack)
            eps_min = min(eps_min, rhs)
            eps_max = max(eps_max, rhs)

    print(f"[{label}] checked {n_checked} points across {len(cells)} cells "
          f"({n_violations} violations, worst-case slack = {worst_slack:.6g})")
    print(f"[{label}] eps_hat min = {eps_min:.6g}, max = {eps_max:.6g}")
    return worst_slack, n_violations


L_M = 1.0  # Lipschitz constant of true_map=max(.) under the sup-norm

print("\nVerifying Eq. (18) on the y-subsystem:")
verify_eps_bound(Data_sety, My, rho_y, L_M, true_map, Ny,
                  n_random=200, label="y-subsystem")

print("\nVerifying Eq. (18) on the z-subsystem:")
verify_eps_bound(Data_setz, Mz, rho_z, L_M, true_map, Nz,
                  n_random=200, label="z-subsystem")

My: [0.1996953 0.1996953 0.1996953 0.1996953]
Mz: [0.49998827 0.49998827]

Verifying Eq. (18) on the y-subsystem:
[y-subsystem] checked 108000 points across 500 cells (0 violations, worst-case slack = 0)
[y-subsystem] eps_hat min = 3.70884, max = 23.7119

Verifying Eq. (18) on the z-subsystem:
[z-subsystem] checked 102000 points across 500 cells (0 violations, worst-case slack = 0)
[z-subsystem] eps_hat min = 1.00001, max = 17.5004


(0.0, 0)